#Setup

In [48]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
WEATHERSTACK_API_KEY = userdata.get('WEATHERSTACK_API_KEY')

#Install the Library

In [49]:
!pip install -qU langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith==0.4.42 langgraph

#check the version

In [50]:
!pip show langchain langchain-groq langchain-community langchain-core requests duckduckgo-search ddgs langsmith langgraph

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.1.2
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: langchain-core
Version: 1.

#Import core libraries

In [51]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
import requests

#Setup In Built Tools

In [52]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()


#Custom Tool(live Weather Tool)

In [53]:
@tool
def get_weather_data(city: str) -> str:
  """
  This tool fetches the live weather data of a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}'

  response = requests.get(url)

  return response.json()



#Initialize the GROQ model

In [54]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key = GROQ_API_KEY,
    temperature = 0,
    model_kwargs={"tool_choice":"auto"}
)

#Import Agent Components

In [55]:
from langchain.agents import create_agent
from langsmith import Client


In [56]:
client = Client()
prompt = client.pull_prompt("hwchase17/react")
prompt_template_string = prompt.template

In [57]:
prompt.template

'Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'

In [58]:
agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt = prompt_template_string
)

In [66]:
response = agent.invoke(
    {"messages": [("user", "Find the captial of Rajasthan, then find its weather condition")]}
)
print(response)

{'messages': [HumanMessage(content='Find the captial of Rajasthan, then find its weather condition', additional_kwargs={}, response_metadata={}, id='2006a1a3-7f46-42cc-af90-88fbca5a53fb'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to answer: Find the capital of Rajasthan, then find its weather condition. Use tools: duckduckgo_search to find capital, and get_weather_data for weather. Likely capital is Jaipur. Then get weather for Jaipur. Use get_weather_data. Let's do search to confirm.", 'tool_calls': [{'id': 'fc_771a0904-9fb1-42ba-9cba-74ef3d4b2cee', 'function': {'arguments': '{"query":"capital of Rajasthan"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 320, 'total_tokens': 411, 'completion_time': 0.189273131, 'completion_tokens_details': {'reasoning_tokens': 59}, 'prompt_time': 0.015020186, 'prompt_tokens_details': None, 'queue_time': 0.054314493, 'total_time': 0.204293

In [67]:
response['messages']

[HumanMessage(content='Find the captial of Rajasthan, then find its weather condition', additional_kwargs={}, response_metadata={}, id='2006a1a3-7f46-42cc-af90-88fbca5a53fb'),
 AIMessage(content='', additional_kwargs={'reasoning_content': "We need to answer: Find the capital of Rajasthan, then find its weather condition. Use tools: duckduckgo_search to find capital, and get_weather_data for weather. Likely capital is Jaipur. Then get weather for Jaipur. Use get_weather_data. Let's do search to confirm.", 'tool_calls': [{'id': 'fc_771a0904-9fb1-42ba-9cba-74ef3d4b2cee', 'function': {'arguments': '{"query":"capital of Rajasthan"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 320, 'total_tokens': 411, 'completion_time': 0.189273131, 'completion_tokens_details': {'reasoning_tokens': 59}, 'prompt_time': 0.015020186, 'prompt_tokens_details': None, 'queue_time': 0.054314493, 'total_time': 0.204293317}, 'model

#Deploy

In [68]:
import gradio as gr

In [69]:
def get_response(query):
  response = agent.invoke(
    {"messages": [("user", query)]}
)

  return response['messages'][-1].content

In [70]:
get_response("Find the capital of Rajasthan, then find its weather condition")

'**Capital of Rajasthan:** Jaipur  \n\n**Current weather in Jaipur (as of 2026‑04‑19 15:57 local time):**  \n- **Condition:** Sunny  \n- **Temperature:** 39\u202f°C (feels like 39\u202f°C)  \n- **Humidity:** 13\u202f%  \n- **Wind:** 23\u202fkm/h from the west (262°)  \n- **Pressure:** 1007\u202fmb  \n- **UV Index:** 4  \n\nSo, the capital of Rajasthan is **Jaipur**, and its present weather is **sunny with a temperature of about 39\u202f°C**.'

In [72]:
iface = gr.Interface(
    fn = get_response,
    inputs = gr.Textbox(
        label = "Ask a question to your AI Agent",
        placeholder = "Eg. Find the capital of Rajasthan, then find its weather condition",
        lines = 2
    ),
    outputs = gr.Textbox(
      label = "Response",
      lines = 10
    ),
    title = "AI Agent with Web Access and live weather Data",
    description = "This AI Agent has access to the internet. You can ask anything"
)

In [73]:
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff8cc6c5649826656d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
